# Predicting the rank of a given League player using Naive Bayes
*What is the average rank of players in a match given features `X` occuring?* 

* Context Fields: `MINUTE`
* Main Features: `KILLS`, `DEATHS`, `ASSISTS`, `CS`, `JUNGLE_CS`, `CURRENT_GOLD`, `TOTAL_GOLD`
* Response: `AVERAGE_RANK` of a match

## Import Statements

In [ ]:
%%sql
USE WAREHOUSE COMPUTE_WH;

USE DATABASE LEAGUE_RECORDS;

USE SCHEMA SILVER;

In [ ]:
from dataclasses import dataclass
from math import factorial, e 
import math
from typing import Literal

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import poisson
from snowflake.snowpark import DataFrame as SnowparkDataFrame

## Source Dataset Schema


In [ ]:
WITH MATCH_MAX_INTERVAL AS (
    SELECT MATCH_ID, MAX(MINUTE) AS END_INTERVAL
    FROM PLAYER_INTERVAL_SILVER
    GROUP BY MATCH_ID
),
PLAYER_STATS_AT_END_INTERVAL AS (
    SELECT P.MATCH_ID, P.PARTICIPANT_POS_ID,
        SUM(P.CS + P.JUNGLE_CS) AS TOTAL_NET_CS,
        SUM(P.KILLS) AS TOTAL_KILLS,
        SUM(P.DEATHS) AS TOTAL_DEATHS,
        SUM(P.ASSISTS) AS TOTAL_ASSISTS
    FROM MATCH_MAX_INTERVAL AS MX
    JOIN PLAYER_INTERVAL_SILVER AS P
        ON P.MATCH_ID = MX.MATCH_ID
        AND P.MINUTE = MX.END_INTERVAL
    GROUP BY P.MATCH_ID, P.PARTICIPANT_POS_ID
)

SELECT 
    -- Context
    MS.MATCH_ID,
    PS.PARTICIPANT_POS_ID,
    -- Filter
    PS.LANE,
    PS.CHAMPION,
    -- LAMBDA
    ROUND(MS.GAME_DURATION / 60, 2) AS GAME_MINUTES,
    ROUND(MS.GAME_DURATION / 60, 2) + 1.5 AS SPAWN_ADJ_GAME_MINUTES,
    PX.TOTAL_NET_CS,
FROM PLAYER_STATS_AT_END_INTERVAL AS PX
JOIN MATCHES_SUMMARY_SILVER AS MS
    ON MS.MATCH_ID = PX.MATCH_ID
JOIN PLAYERS_SUMMARY_SILVER AS PS
    ON PS.MATCH_ID = PX.MATCH_ID
    AND PS.PARTICIPANT_POS_ID = PX.PARTICIPANT_POS_ID
;

In [ ]:
def poisson_pmf(ld: float, x: int) -> float:
    log_p = -ld + x * math.log(ld) - math.lgamma(x + 1)
    
    return math.exp(log_p)

In [ ]:
def poisson_probability_of_kills(
    # ----- Base data
    data: SnowparkDataFrame,
    lambda_var: str,
    lambda_dur: str,
    # ----- Filter by
    lane: str = None,
    champion: str = None,
    # ----- Predict for new value
    x: int = 3,
    k: int = 5,
    cdf_direction: Literal['le', 'ge'] = None,
) -> None:
    print(f"Poisson query: Calculating probability of achieving {x} {lambda_var} at {k} minutes.")

    df = data.to_pandas()
    if lane is not None:
        df = df.query("LANE == @lane")
    if champion is not None:
        df = df.query("CHAMPION == @champion")

    if df.empty:
        print(f"No data found for lane={lane}, champion={champion}. Aborting.")
        return
    
    mean_cs = df[lambda_var].mean()
    var_cs = df[lambda_var].var()
    print(f"Mean: {mean_cs:.2f}, Variance: {var_cs:.2f}, Ratio: {var_cs/mean_cs:.2f}")

    ld = np.mean(df[lambda_var] / df[lambda_dur])
    print(f"The base poisson distribution for {lambda_var} per {lambda_dur} has lambda --> {ld}")

    kld = k * ld
    print(f"The poisson distribution for {lambda_var} every {k} {lambda_dur} has lambda --> {kld}")

    if cdf_direction:
        poisson_cdf = 0
        upper = x if cdf_direction == 'le' else x - 1
        for n in range(upper + 1):
            poisson_cdf += poisson_pmf(kld, n)
        p_x = poisson_cdf
        if cdf_direction == 'ge':
            p_x = 1 - p_x
    else:
        p_x = poisson_pmf(kld, x)

    fmt_p_x = p_x * 100

    print(f"Probability of achieving {cdf_direction} {x} {lambda_var} at {k} minutes, at base lambda = {ld}")
    print(f"---> {fmt_p_x:.2f}%")

    # ----- Visualization
    upper_bound = max(int(kld + 4 * np.sqrt(kld)) + 1, x + 5)
    x_arr = np.arange(0, upper_bound)
    y_arr = [poisson_pmf(kld, n) for n in x_arr]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(x_arr, y_arr, color="steelblue")
    ax.set_title(f"Poisson distribution of {lambda_var} within {k} {lambda_dur}")
    ax.set_xlabel(lambda_var)
    ax.set_ylabel("P")
    
    direction_label = cdf_direction if cdf_direction else "="
    ax.axvline(kld, color="black", linestyle="-", linewidth=1.5, label=f"λ = {kld:.2f}")
    ax.axvline(x, color="red", linestyle="--", linewidth=1.5, label=f"P(X {direction_label} {x}) = {p_x:.2f}")
    ax.legend()
    
    plt.show()
    plt.close(fig)

In [ ]:
poisson_probability_of_kills(
    data=per_player_avg_cs,
    lambda_var='TOTAL_NET_CS',
    lambda_dur='SPAWN_ADJ_GAME_MINUTES',
    # ----- Filter by
    champion='Jhin',
    # -----
    x=120,
    k=15,
    cdf_direction="ge"
)

In [ ]:
def prior_prob_distribution(
    df: pd.DataFrame,
    group: str
) -> pd.DataFrame:
    return (df
        .reset_index(names="id")
        .groupby(group, as_index=False)
        .agg(p=(
            "id", 
            lambda x: x.count() / len(df)
        ))
        .sort_values("p", ascending=False)
    )

In [ ]:
def feature_dist_given_group(
    df: pd.DataFrame,
    feature: str,
    group_var: str, # Average Rank
    group_value: str, # Master, Grandmaster, etc.
    show_plots: bool = True
) -> tuple[float, float]:
    arr = (df
        .loc[df[group_var] == group_value, feature]
        .to_numpy()
    )
    arr_mean = np.mean(arr)
    arr_std = np.std(arr)
    
    if show_plots:
        fig, ax = plt.subplots()
        sns.histplot(arr, ax=ax)

        ax.set_title(f"Distribution of {feature} for {group_value} {group_var} matches.")
        ax.set_xlabel(feature)
        
        ax.axvline(arr_mean, color="black", linestyle="-", linewidth=1.5, label=f"mean = {arr_mean:.2f}")
        ax.axvline(arr_mean - arr_std, color="gray", linestyle="--", linewidth=1, label=f"std = {arr_std:.2f}")
        ax.axvline(arr_mean + arr_std, color="gray", linestyle="--", linewidth=1)
        
        plt.show()
        plt.close(fig)

    return arr_mean, arr_std

In [ ]:
feature_dist_given_group(
    df=ranked_match_summary.to_pandas(),
    feature="AVG_CS_PER_MINUTE",
    group_var="AVERAGE_RANK",
    group_value="Master",
    show_plots=True
)

In [ ]:
feature_dist_given_group(
    df=ranked_match_summary.to_pandas(),
    feature="TOTAL_DEATHS",
    group_var="AVERAGE_RANK",
    group_value="Master",
    show_plots=True
)

In [ ]:
def predict_many_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    response: str,
    features: list[str],
) -> pd.DataFrame:
    # ----- Prior from training data
    prior = prior_prob_distribution(df=train_df, group=response)
    print(prior)

    # ----- Per-rank, per-feature mean/std and log-prior, fit once on training data
    rank_params = {}
    for row in prior.itertuples():
        rank = getattr(row, response)
        p_prior = getattr(row, "p")

        feature_stats = {}
        for feature in features:
            mean, std = feature_dist_given_group(
                df=train_df,
                feature=feature,
                group_var=response,
                group_value=rank,
                show_plots=False,
            )
            feature_stats[feature] = {"mean": mean, "std": std}

        rank_params[rank] = {
            "log_prior": np.log(p_prior),
            "features": feature_stats,
        }

    # ----- Score every test row against every rank, summing log-likelihoods across features
    scores = pd.DataFrame(index=test_df.index)

    for rank, params in rank_params.items():
        rank_score = np.full(len(test_df), params["log_prior"])

        for feature in features:
            test_values = test_df[feature].to_numpy()
            stats = params["features"][feature]

            likelihood = norm.pdf(test_values, loc=stats["mean"], scale=stats["std"])
            likelihood = np.clip(likelihood, 1e-300, None)

            rank_score = rank_score + np.log(likelihood)

        scores[rank] = rank_score

    # ----- Pick the rank with the highest score per row
    predicted = scores.idxmax(axis=1)

    # ----- Assemble comparison dataframe
    result = test_df[features + [response]].copy()
    result["predicted"] = predicted.values
    result = result.rename(columns={response: "actual"})
    result["correct"] = result["actual"] == result["predicted"]

    return result

In [ ]:
HIGH_ELO_RANKS = {"Master", "Grandmaster", "Challenger"}

def add_elo_tier(df: pd.DataFrame, rank_col: str = "AVERAGE_RANK") -> pd.DataFrame:
    df = df.copy()
    df["ELO_TIER"] = df[rank_col].apply(
        lambda r: "High Elo" if r in HIGH_ELO_RANKS else "Low Elo"
    )
    return df

In [ ]:
def gaussian_nb(
    data: SnowparkDataFrame,
    response: str,
    features: list[str],
) -> pd.DataFrame:
    df = data.to_pandas()
    df = add_elo_tier(df, rank_col=response)

    train_df, test_df = split_test_train(
        df=df[features + ["ELO_TIER"]],
        pct=0.8,
        random_state=42,
    )

    results = predict_many_features(
        train_df=train_df,
        test_df=test_df,
        response="ELO_TIER",
        features=features,
    )

    accuracy = results["correct"].mean()
    print(f"Accuracy: {accuracy:.4f}")
    print(results.head(20))
    print("\nConfusion breakdown:")
    print(pd.crosstab(results["actual"], results["predicted"]))

    return results

In [ ]:
gaussian_nb(
    ranked_match_summary,
    response="AVERAGE_RANK",
    features=["AVG_CS_PER_MINUTE", "AVG_GOLD_PER_MINUTE", "AVG_KDA_PER_MINUTE"]
)